# WildfireSpreadTS — download, extract, and validate format on Kaggle

Diagnostic run before building the real training pipeline. Goals:
1. Confirm disk layout: download/extract into `/kaggle/temp` (ephemeral, no 20GB output cap), not `/kaggle/working`.
2. Download `WildfireSpreadTS.zip` (48.4 GB, Zenodo record 8006177, CC BY 4.0).
3. Clone the official repo and use ITS OWN PyTorch Dataset/DataModule to load one real sample.
4. Report actual tensor shape / channel count / value ranges -- do not assume, verify.

This does NOT train anything yet. It only validates that the real data is
usable before we build a schema-mapping bridge into WildfireConvLSTM.

In [ ]:
import os, subprocess

print('--- disk before download ---')
os.system('df -h /kaggle/working /kaggle/temp 2>&1 || df -h /kaggle')

TEMP = '/kaggle/temp/wfts'
os.makedirs(TEMP, exist_ok=True)
print('Using scratch dir:', TEMP)

In [ ]:
# Clone the official repo for its PyTorch Dataset / Lightning DataModule
!git clone --depth 1 https://github.com/SebastianGer/WildfireSpreadTS.git /kaggle/working/wfts_repo
!pip install -q lightning h5py rasterio 2>&1 | tail -5

In [ ]:
# Download the dataset zip straight into ephemeral /kaggle/temp (NOT /kaggle/working).
# Using a manual chunked download with flushed print() every 500MB instead of
# wget's carriage-return progress bar, which does not reliably show up in
# Kaggle's captured kernel logs -- this way progress is actually visible.
import time
import urllib.request

url = 'https://zenodo.org/records/8006177/files/WildfireSpreadTS.zip?download=1'
dest = f'{TEMP}/WildfireSpreadTS.zip'

t0 = time.time()
chunk_size = 1 << 20  # 1 MB
report_every = 500  # MB
downloaded = 0

with urllib.request.urlopen(url) as resp, open(dest, 'wb') as f:
    total = int(resp.headers.get('Content-Length', 0))
    print(f'Total size: {total / 1e9:.2f} GB', flush=True)
    last_report_mb = 0
    while True:
        chunk = resp.read(chunk_size)
        if not chunk:
            break
        f.write(chunk)
        downloaded += len(chunk)
        mb = downloaded // (1 << 20)
        if mb - last_report_mb >= report_every:
            elapsed = time.time() - t0
            rate = downloaded / elapsed / 1e6 if elapsed > 0 else 0
            pct = 100 * downloaded / total if total else 0
            print(f'  {mb} MB downloaded ({pct:.1f}%), {rate:.1f} MB/s, {elapsed:.0f}s elapsed', flush=True)
            last_report_mb = mb

print(f'Download complete: {downloaded / 1e9:.2f} GB in {time.time() - t0:.0f}s', flush=True)
os.system(f'ls -la {dest}')
os.system('df -h /kaggle/temp')

In [ ]:
# Verify checksum before spending time extracting a bad download
expected_md5 = 'dc1a04e63ccc70037b277d585b8fe761'
import hashlib
h = hashlib.md5()
with open(dest, 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
actual = h.hexdigest()
print('expected:', expected_md5)
print('actual:  ', actual)
assert actual == expected_md5, 'Checksum mismatch -- re-download before proceeding.'
print('OK: checksum verified.')

In [ ]:
# Extract into /kaggle/temp as well (extracted contents will exceed 20GB)
extract_dir = f'{TEMP}/extracted'
os.makedirs(extract_dir, exist_ok=True)
rc = os.system(f'unzip -q {dest} -d {extract_dir}')
print('unzip exit code:', rc)
os.system(f'du -sh {extract_dir}')
os.system(f'find {extract_dir} -maxdepth 2 | head -40')

In [ ]:
# Use the OFFICIAL dataset class, not a guessed format, to load one real sample
import sys
sys.path.insert(0, '/kaggle/working/wfts_repo/src')

# The exact import path depends on the repo layout discovered above --
# print the repo tree first so we know what to import.
os.system('find /kaggle/working/wfts_repo -iname "*.py" | grep -i -E "dataset|datamodule" ')

In [ ]:
# NOTE: fill in the actual class import once the previous cell's output is known,
# then load one sample and print its real shape/channels/dtype/value range.
# Placeholder left intentionally -- do not guess the API surface blind.
print('Repo structure discovered above. Next: import the real Dataset class and load one sample.')